In [2]:
import numpy as np
import pandas as pd
from pulp import *
import copy

In [3]:
data = pd.read_csv('dataset.csv')
data = data.sort_values(by = 'Unnamed: 0')
ids = data[['Unnamed: 0']]
data = data.drop('Unnamed: 0', axis = 'columns')
data = data.reset_index(drop = True)
data.index = data.index+1
data.index = 'a'+ data.index.astype('str')
data

,Rooms,Price,Distance,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt
a1,3,840000.0,13.8,3.0,2.0,2.0,534.0,136.00,1965.0
a2,4,1100000.0,5.9,4.0,2.0,5.0,559.0,195.00,1920.0
a3,2,495000.0,13.9,2.0,1.0,1.0,76.0,79.00,1980.0
a4,3,1120000.0,7.8,3.0,2.0,1.0,293.0,180.00,2006.0
a5,4,2325000.0,9.2,4.0,3.0,2.0,638.0,314.00,1930.0
a6,3,822000.0,13.0,3.0,1.0,4.0,700.0,105.00,1950.0
a7,3,1560000.0,4.6,3.0,2.0,0.0,198.0,148.00,1910.0
a8,2,650000.0,10.5,2.0,1.0,3.0,620.0,85.00,1950.0
a9,3,1223500.0,7.9,3.0,2.0,2.0,721.0,136.00,1980.0
a10,2,790000.0,11.2,2.0,1.0,2.0,196.0,109.00,1970.0


In [3]:
data.describe()

,Rooms,Price,Distance,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt
count,50.000000,5.000000e+01,50.000000,50.000000,50.000000,50.00000,50.000000,50.000000,50.000000
mean,2.940000,1.010540e+06,10.760000,2.920000,1.580000,1.72000,400.440000,146.037400,1966.520000
std,0.977502,4.836915e+05,7.117727,0.965528,0.641745,1.03095,305.460586,67.062725,33.749522
min,1.000000,3.610000e+05,2.100000,1.000000,1.000000,0.00000,0.000000,46.000000,1890.000000
25%,2.000000,6.507500e+05,7.325000,2.000000,1.000000,1.00000,170.000000,105.250000,1942.500000
50%,3.000000,8.785000e+05,9.350000,3.000000,1.500000,2.00000,318.500000,132.000000,1970.000000
75%,3.000000,1.255000e+06,13.975000,3.000000,2.000000,2.00000,599.250000,183.750000,1992.750000
max,6.000000,2.440000e+06,45.900000,6.000000,3.000000,5.00000,1452.000000,314.000000,2013.000000


## **1. UTA algorithm - finding minimal subset of inconsistent constraints** ##

In [227]:
# preference_list - list of pairs (better, worse)
preference_list = [("a4", "a9"), ("a9", "a13"), ("a13", "a4"), ("a4", "a1"), ("a45", "a32"), 
("a27", "a18"), ("a25", "a28"), ("a45", "a50"), ("a43", "a37"), ("a39", "a38"), 
("a31", "a33"), ("a30", "a42"), ("a40", "a35"), ("a27", "a26"), ("a3", "a11"), 
("a6", "a7"), ("a21", "a2"), ("a15", "a12"), ("a20", "a29"), ("a41", "a24"), 
("a23", "a34"), ("a5", "a18"), ("a8", "a14"), ("a10", "a19"), ("a22", "a16"),
("a49", "a44"), ("a36", "a48"), ("a47", "a46"),("a1", "a13"), ("a21", "a12"),
("a27", "a17"), ("a43", "a28"), ("a50", "a49"), ("a25", "a16"), ("a4", "a16"),
("a8", "a11"), ("a20", "a13"), ("a26", "a5"), ("a32", "a33"), ("a40", "a33"),
("a8", "a6"), ("a1", "a11"), ("a11", "a13"), ("a45", "a41"), ("a21", "a23"),
("a18", "a7"), ("a24", "a19"), ("a27", "a21"), ("a39", "a30"), ("a49", "a23")]

# Inconsistencies: cycle a4, a9, a13

In [228]:
# criteria_types - list of pairs in form (gain or cost type, is discrete?)
criteria_types = [('gain', True), ('cost', False), 
('cost', False), ('gain', True), ('gain', True), 
('gain', True), ('gain', False), ('gain', False), 
('gain', False)]

In [213]:
def build_uta_model(preference_list, df, criteria_types):
    # 0. Define Linear Problem - we want to minimize number of const that are excluded
    model = LpProblem('uta_model', LpMinimize)
    eps = 0.001

    pairwise_comparisons = copy.deepcopy(preference_list)
    binary_variables = LpVariable.dicts('bin_var', pairwise_comparisons, cat='Binary')
    model += lpSum(list(binary_variables.values()))

    # 1. Add weights const
    weights = []
    for crit in df.columns:
        weights.append(LpVariable(f'w_{crit}', lowBound = 1/(5*len(df.columns)), upBound = 0.5, cat = 'Continuous'))

    # 2. Add norm const
    model += lpSum(weights) == 1

    # 3. Utility constraints
    utility_vars = {}
    for alt in df.index:
        for criterion in df.columns:
            curr_util_name = 'u_'+alt+'_'+criterion
            utility_vars[curr_util_name] = LpVariable(curr_util_name, lowBound=0, cat = 'Continuous')
    
    for i in range(len(criteria_types)):
        # Find best and worst
        if criteria_types[i][0] == 'cost':
            best_idx = np.argmin(df[df.columns[i]])
            worst_idx = np.argmax(df[df.columns[i]])
        else:
            worst_idx = np.argmin(df[df.columns[i]])
            best_idx = np.argmax(df[df.columns[i]])

        model += utility_vars[f'u_{df.index[best_idx]}_{df.columns[i]}'] == 1
        model += utility_vars[f'u_{df.index[worst_idx]}_{df.columns[i]}'] == 0

        # Monocity constraint
        if criteria_types[i][0] == 'cost':
            sorted_alts = np.argsort(df[df.columns[i]])
        else:
            sorted_alts = np.argsort(df[df.columns[i]])[::-1]
        for j in range(len(sorted_alts)-1):
                model += utility_vars[f'u_{df.index[sorted_alts[j]]}_{df.columns[i]}'] >= utility_vars[f'u_{df.index[sorted_alts[j+1]]}_{df.columns[i]}']
    print(model)

    # 4. Add utility functions
    for better, worse in pairwise_comparisons:
        # if criterion can't be met, then 1 is substracted on the right side so that it holds
        model += lpSum([weights[i] * utility_vars[f'u_{better}_{df.columns[i]}'] for i in range(len(weights))]) > lpSum([weights[i] * utility_vars[f'u_{worse}_{df.columns[i]}'] for i in range(len(weights))]) - binary_vars[(alt1, alt2)] + eps
    print(model)
    return model

In [214]:
def build_uta_model_2_1(preference_list, df, criteria_types):
    model = LpProblem('uta_model', LpMinimize)
    eps = 0.001

    # 0. Add binary variables for each pairwise preference
    pairwise_comparisons = copy.deepcopy(preference_list)
    binary_variables = LpVariable.dicts('bin_var', pairwise_comparisons, cat='Binary')
    model += lpSum(list(binary_variables.values()))

    # 1. Add weights
    weights = {}
    n_criteria = len(df.columns)
    for crit in df.columns:
        weights[crit] = LpVariable(f'w_{crit}', lowBound=1/(5*n_criteria), upBound=0.5, cat='Continuous')

    # 2. Add normalization constraint
    model += lpSum(weights.values()) == 1

    # 3. Utility values for each alternative per criterion (constants here)
    # Normalize criterion values between 0 and 1
    normalized_df = df.copy()
    for crit, (ctype, isDiscrete) in zip(df.columns, criteria_types):
        if ctype == 'cost':
            normalized_df[crit] = (df[crit].max() - df[crit]) / (df[crit].max() - df[crit].min())
        else:
            normalized_df[crit] = (df[crit] - df[crit].min()) / (df[crit].max() - df[crit].min())

    # Total utility for each alternative
    U = {}
    for alt in df.index:
        U[alt] = LpVariable(f'U_{alt}', lowBound=0, upBound=1, cat='Continuous')
        model += U[alt] == lpSum([weights[crit] * normalized_df.loc[alt, crit] for crit in df.columns])

    # 4. Pairwise constraints (some may be rejected by binary variables)
    for (alt1, alt2) in pairwise_comparisons:
        model += U[alt1] >= U[alt2] + eps - binary_variables[(alt1, alt2)]

    return model

In [229]:
def compute_breakpoints(values, num_breakpoints):
    """Compute equally spaced breakpoints over a criterion’s value range."""
    min_val, max_val = np.min(values), np.max(values)
    return np.linspace(min_val, max_val, num_breakpoints)

In [230]:
def build_uta_model_with_piecewise(preference_list, df, criteria_types, num_breakpoints=6):
    model = LpProblem('UTA_Model_Piecewise', LpMinimize)
    eps = 0.01

    alternatives = df.index.tolist()
    criteria = df.columns.tolist()
    n_criteria = len(criteria)

    # 1a Define weights
    weights = {
        crit: LpVariable(f'w_{crit}', lowBound=1 / (5 * n_criteria), upBound=0.5, cat='Continuous')
        for crit in criteria
    }

    # 1b Normalization constraint
    model += lpSum([weights[i] for i in df.columns]) == 1

    # 2. Define breakpoints and lambda values
    breakpoints = {crit: compute_breakpoints(df[crit], num_breakpoints) for crit in criteria}
    lambdas = {
        (crit, k): LpVariable(f'lambda_{crit}_{k}', lowBound=0, upBound=1, cat='Continuous')
        for crit in criteria for k in range(num_breakpoints)
    }

    for i in range(len(criteria_types)):
        if criteria_types[i][0] == 'cost':
            model += lambdas[(criteria[i], 0)] == 1
            model += lambdas[(criteria[i], num_breakpoints - 1)] == 0
            for k in range(num_breakpoints - 1):
                model += lambdas[(criteria[i], k)] >= lambdas[(criteria[i], k + 1)]
        else:
            model += lambdas[(criteria[i], 0)] == 0
            model += lambdas[(criteria[i], num_breakpoints - 1)] == 1
            for k in range(num_breakpoints - 1):
                model += lambdas[(criteria[i], k)] <= lambdas[(criteria[i], k + 1)]

    # 3. Interpolation weights
    interpolation_weights = {}
    for alt in alternatives:
        for crit in criteria:
            val = df.loc[alt, crit]
            bp = breakpoints[crit]
            for k in range(num_breakpoints - 1):
                if bp[k] <= val <= bp[k + 1]:
                    alpha = (val - bp[k]) / (bp[k + 1] - bp[k]) if bp[k + 1] != bp[k] else 0
                    interpolation_weights[(alt, crit)] = (k, 1 - alpha, alpha)
                    break

    # 4. Define U(alt) as sum of weighted marginal utilities
    U = {
        alt: LpVariable(f'U_{alt}', lowBound=0, upBound=1, cat='Continuous')
        for alt in alternatives
    }

    for alt in alternatives:
        weighted_sum_expr = []
        for crit in criteria:
            k, beta1, beta2 = interpolation_weights[(alt, crit)]
            marginal = beta1 * lambdas[(crit, k)] + beta2 * lambdas[(crit, k + 1)]

            # New variable for marginal utility
            u_crit = LpVariable(f'u_{alt}_{crit}', lowBound=0, upBound=1)
            model += u_crit == marginal

            # New variable for weighted utility
            wu = LpVariable(f'wu_{alt}_{crit}', lowBound=0, upBound=1)

            # Linearization of wu = w * u
            model += wu <= weights[crit]
            model += wu <= u_crit
            model += wu >= weights[crit] + u_crit - 1
            # model += wu >= 0

            weighted_sum_expr.append(wu)

        model += U[alt] == lpSum(weighted_sum_expr)

    # 5. Preference constraints with slack variables
    binary_vars = {
        (a1, a2): LpVariable(f'z_{a1}_{a2}', cat='Binary')
        for (a1, a2) in preference_list
    }
    model += lpSum(binary_vars.values())  # objective

    for (a1, a2) in preference_list:
        model += U[a1] >= U[a2] + eps - binary_vars[(a1, a2)]
    return model

In [231]:
model = build_uta_model_with_piecewise(preference_list, data, criteria_types)
status = model.solve(solver = GLPK())

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --cpxlp /var/folders/8m/nx_b_wh17dg77b9kb95gxng80000gp/T/ad460e4d478b4dcd9980992a00e4632a-pulp.lp
 -o /var/folders/8m/nx_b_wh17dg77b9kb95gxng80000gp/T/ad460e4d478b4dcd9980992a00e4632a-pulp.sol
Reading problem data from '/var/folders/8m/nx_b_wh17dg77b9kb95gxng80000gp/T/ad460e4d478b4dcd9980992a00e4632a-pulp.lp'...
1964 rows, 1063 columns, 5073 non-zeros
50 integer variables, all of which are binary
3378 lines were read
GLPK Integer Optimizer 5.0
1964 rows, 1063 columns, 5073 non-zeros
50 integer variables, all of which are binary
Preprocessing...
1715 rows, 944 columns, 4390 non-zeros
50 integer variables, all of which are binary
Scaling...
 A: min|aij| =  4.566e-03  max|aij| =  1.000e+00  ratio =  2.190e+02
GM: min|aij| =  1.198e-01  max|aij| =  8.344e+00  ratio =  6.962e+01
EQ: min|aij| =  1.436e-02  max|aij| =  1.000e+00  ratio =  6.962e+01
2N: min|aij| =  7.937e-03  max|aij| =  1.727e+00  ratio =  2.176e+02
C

In [232]:
print("status:", model.status, LpStatus[model.status])
print("objective:", model.objective.value())

status: 1 Optimal
objective: 1


In [233]:
for var in model.variables():
    if 'z_' in var.name and var.value() > 0:
        print(var.name, var.value(), "is inconsistent")
    # else:
    #     print(var.name, var.value())

z_a13_a4 1 is inconsistent


## **UTA 2.2** ##

In [234]:
consistent_preference_list = list(set(preference_list).difference({('a13', 'a4')}))

In [235]:
consistent_preference_list

[('a31', 'a33'),
 ('a40', 'a35'),
 ('a45', 'a32'),
 ('a49', 'a23'),
 ('a39', 'a38'),
 ('a3', 'a11'),
 ('a5', 'a18'),
 ('a47', 'a46'),
 ('a45', 'a50'),
 ('a15', 'a12'),
 ('a30', 'a42'),
 ('a4', 'a1'),
 ('a6', 'a7'),
 ('a27', 'a17'),
 ('a27', 'a26'),
 ('a18', 'a7'),
 ('a21', 'a12'),
 ('a21', 'a2'),
 ('a4', 'a9'),
 ('a20', 'a13'),
 ('a21', 'a23'),
 ('a50', 'a49'),
 ('a36', 'a48'),
 ('a40', 'a33'),
 ('a1', 'a11'),
 ('a10', 'a19'),
 ('a24', 'a19'),
 ('a25', 'a16'),
 ('a20', 'a29'),
 ('a43', 'a37'),
 ('a11', 'a13'),
 ('a23', 'a34'),
 ('a8', 'a11'),
 ('a4', 'a16'),
 ('a39', 'a30'),
 ('a32', 'a33'),
 ('a9', 'a13'),
 ('a27', 'a18'),
 ('a8', 'a6'),
 ('a1', 'a13'),
 ('a43', 'a28'),
 ('a45', 'a41'),
 ('a22', 'a16'),
 ('a26', 'a5'),
 ('a25', 'a28'),
 ('a49', 'a44'),
 ('a8', 'a14'),
 ('a27', 'a21'),
 ('a41', 'a24')]

In [236]:
def build_uta_model_2_2(preference_list, df, criteria_types, num_breakpoints = 6):
    model = LpProblem('UTA_Model_max_distance', LpMaximize)
    eps = 0.001

    alternatives = df.index.tolist()
    criteria = df.columns.tolist()
    n_criteria = len(criteria)

    # 1a Define weights
    weights = {
        crit: LpVariable(f'w_{crit}', lowBound=1 / (5 * n_criteria), upBound=0.5, cat='Continuous')
        for crit in criteria
    }

    # 1b Normalization constraint
    model += lpSum([weights[i] for i in df.columns]) == 1

    # 2. Define breakpoints and lambda values
    breakpoints = {crit: compute_breakpoints(df[crit], num_breakpoints) for crit in criteria}
    lambdas = {
        (crit, k): LpVariable(f'lambda_{crit}_{k}', lowBound=0, upBound=1, cat='Continuous')
        for crit in criteria for k in range(num_breakpoints)
    }

    for i in range(len(criteria_types)):
        if criteria_types[i][0] == 'cost':
            model += lambdas[(criteria[i], 0)] == 1
            model += lambdas[(criteria[i], num_breakpoints - 1)] == 0
            for k in range(num_breakpoints - 1):
                model += lambdas[(criteria[i], k)] >= lambdas[(criteria[i], k + 1)] + 0.1
        else:
            model += lambdas[(criteria[i], 0)] == 0
            model += lambdas[(criteria[i], num_breakpoints - 1)] == 1
            for k in range(num_breakpoints - 1):
                model += lambdas[(criteria[i], k)] + 0.1 <= lambdas[(criteria[i], k + 1)]

    # 3. Interpolation weights
    interpolation_weights = {}
    for alt in alternatives:
        for crit in criteria:
            val = df.loc[alt, crit]
            bp = breakpoints[crit]
            for k in range(num_breakpoints - 1):
                if bp[k] <= val <= bp[k + 1]:
                    alpha = (val - bp[k]) / (bp[k + 1] - bp[k]) if bp[k + 1] != bp[k] else 0
                    interpolation_weights[(alt, crit)] = (k, 1 - alpha, alpha)
                    break

    # 4. Define U(alt) as sum of weighted marginal utilities
    U = {
        alt: LpVariable(f'U_{alt}', lowBound=0, upBound=1, cat='Continuous')
        for alt in alternatives
    }

    for alt in alternatives:
        weighted_sum_expr = []
        for crit in criteria:
            k, beta1, beta2 = interpolation_weights[(alt, crit)]
            marginal = beta1 * lambdas[(crit, k)] + beta2 * lambdas[(crit, k + 1)]

            # New variable for marginal utility
            u_crit = LpVariable(f'u_{alt}_{crit}', lowBound=0, upBound=1)
            model += u_crit == marginal

            # New variable for weighted utility
            wu = LpVariable(f'wu_{alt}_{crit}', lowBound=0, upBound=1)

            # Linearization of wu = w * u
            model += wu <= weights[crit]
            model += wu <= u_crit
            model += wu >= weights[crit] + u_crit - 1
            # model += wu >= 0

            weighted_sum_expr.append(wu)

        model += U[alt] == lpSum(weighted_sum_expr)

    # 5. Objective function:
    obj = []
    for (a1, a2) in preference_list:
        obj.append(U[a1] - U[a2])
    model += lpSum(obj) # objective

    # 6. Preference relations
    for (a1, a2) in preference_list:
        model += U[a1] >= U[a2] + eps

    return model

In [237]:
model2 = build_uta_model_2_2(consistent_preference_list, data, criteria_types)
status = model2.solve(solver = GLPK())

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --cpxlp /var/folders/8m/nx_b_wh17dg77b9kb95gxng80000gp/T/0fcb84de154f461bb84ecc4b16c30410-pulp.lp
 -o /var/folders/8m/nx_b_wh17dg77b9kb95gxng80000gp/T/0fcb84de154f461bb84ecc4b16c30410-pulp.sol
Reading problem data from '/var/folders/8m/nx_b_wh17dg77b9kb95gxng80000gp/T/0fcb84de154f461bb84ecc4b16c30410-pulp.lp'...
1963 rows, 1013 columns, 5021 non-zeros
3324 lines were read
GLPK Simplex Optimizer 5.0
1963 rows, 1013 columns, 5021 non-zeros
Preprocessing...
1725 rows, 894 columns, 4371 non-zeros
Scaling...
 A: min|aij| =  4.566e-03  max|aij| =  1.000e+00  ratio =  2.190e+02
GM: min|aij| =  1.198e-01  max|aij| =  8.344e+00  ratio =  6.962e+01
EQ: min|aij| =  1.436e-02  max|aij| =  1.000e+00  ratio =  6.962e+01
Constructing initial basis...
Size of triangular part is 1725
      0: obj =  -0.000000000e+00 inf =   1.268e+01 (103)
    111: obj =   3.987395664e+00 inf =   1.249e-16 (0)
*   336: obj =   3.269221745e+01 i

In [238]:
print("status:", model2.status, LpStatus[model2.status])
print("objective:", model2.objective.value())

status: 1 Optimal
objective: 32.69221980000001


In [239]:
for var in model2.variables():
    print(var.name, var.value())

U_a1 0.999
U_a10 0.977778
U_a11 0.001
U_a12 0.0
U_a13 0.0
U_a14 0.0
U_a15 0.977778
U_a16 0.0
U_a17 0.0
U_a18 0.0710027
U_a19 0.0
U_a2 0.4
U_a20 0.977778
U_a21 0.999
U_a22 0.977778
U_a23 0.001
U_a24 0.001
U_a25 0.977778
U_a26 0.26236
U_a27 1.0
U_a28 0.0164502
U_a29 0.0
U_a3 0.977778
U_a30 0.001
U_a31 0.977778
U_a32 0.243902
U_a33 0.0
U_a34 0.0
U_a35 0.0487805
U_a36 0.977778
U_a37 0.0
U_a38 0.0
U_a39 1.0
U_a4 1.0
U_a40 1.0
U_a41 0.002
U_a42 0.0
U_a43 0.799129
U_a44 0.0
U_a45 1.0
U_a46 0.0
U_a47 0.977778
U_a48 0.0342466
U_a49 0.998
U_a5 0.26136
U_a50 0.999
U_a6 0.343902
U_a7 0.0
U_a8 0.977778
U_a9 0.243902
lambda_Bathroom_0 0.0
lambda_Bathroom_1 0.1
lambda_Bathroom_2 0.2
lambda_Bathroom_3 0.3
lambda_Bathroom_4 0.4
lambda_Bathroom_5 1.0
lambda_Bedroom2_0 0.0
lambda_Bedroom2_1 0.1
lambda_Bedroom2_2 0.2
lambda_Bedroom2_3 0.3
lambda_Bedroom2_4 0.4
lambda_Bedroom2_5 1.0
lambda_BuildingArea_0 0.0
lambda_BuildingArea_1 0.586047
lambda_BuildingArea_2 0.686047
lambda_BuildingArea_3 0.786047
lambda

In [240]:
utilities = []
for var in model2.variables():
    if 'U_' in var.name:
        utilities.append((var.name, var.value()))
utilities.sort(key = lambda x: x[1])
utilities

[('U_a12', 0.0),
 ('U_a13', 0.0),
 ('U_a14', 0.0),
 ('U_a16', 0.0),
 ('U_a17', 0.0),
 ('U_a19', 0.0),
 ('U_a29', 0.0),
 ('U_a33', 0.0),
 ('U_a34', 0.0),
 ('U_a37', 0.0),
 ('U_a38', 0.0),
 ('U_a42', 0.0),
 ('U_a44', 0.0),
 ('U_a46', 0.0),
 ('U_a7', 0.0),
 ('U_a11', 0.001),
 ('U_a23', 0.001),
 ('U_a24', 0.001),
 ('U_a30', 0.001),
 ('U_a41', 0.002),
 ('U_a28', 0.0164502),
 ('U_a48', 0.0342466),
 ('U_a35', 0.0487805),
 ('U_a18', 0.0710027),
 ('U_a32', 0.243902),
 ('U_a9', 0.243902),
 ('U_a5', 0.26136),
 ('U_a26', 0.26236),
 ('U_a6', 0.343902),
 ('U_a2', 0.4),
 ('U_a43', 0.799129),
 ('U_a10', 0.977778),
 ('U_a15', 0.977778),
 ('U_a20', 0.977778),
 ('U_a22', 0.977778),
 ('U_a25', 0.977778),
 ('U_a3', 0.977778),
 ('U_a31', 0.977778),
 ('U_a36', 0.977778),
 ('U_a47', 0.977778),
 ('U_a8', 0.977778),
 ('U_a49', 0.998),
 ('U_a1', 0.999),
 ('U_a21', 0.999),
 ('U_a50', 0.999),
 ('U_a27', 1.0),
 ('U_a39', 1.0),
 ('U_a4', 1.0),
 ('U_a40', 1.0),
 ('U_a45', 1.0)]

In [241]:
lambdas = [] # lambdas - values at breakpoints
for var in model2.variables():
    if 'lambda' in var.name:
        lambdas.append((var.name, var.value()))
lambdas.sort(key = lambda x: x[0])
lambdas

[('lambda_Bathroom_0', 0.0),
 ('lambda_Bathroom_1', 0.1),
 ('lambda_Bathroom_2', 0.2),
 ('lambda_Bathroom_3', 0.3),
 ('lambda_Bathroom_4', 0.4),
 ('lambda_Bathroom_5', 1.0),
 ('lambda_Bedroom2_0', 0.0),
 ('lambda_Bedroom2_1', 0.1),
 ('lambda_Bedroom2_2', 0.2),
 ('lambda_Bedroom2_3', 0.3),
 ('lambda_Bedroom2_4', 0.4),
 ('lambda_Bedroom2_5', 1.0),
 ('lambda_BuildingArea_0', 0.0),
 ('lambda_BuildingArea_1', 0.586047),
 ('lambda_BuildingArea_2', 0.686047),
 ('lambda_BuildingArea_3', 0.786047),
 ('lambda_BuildingArea_4', 0.886047),
 ('lambda_BuildingArea_5', 1.0),
 ('lambda_Car_0', 0.0),
 ('lambda_Car_1', 0.4),
 ('lambda_Car_2', 0.5),
 ('lambda_Car_3', 0.6),
 ('lambda_Car_4', 0.7),
 ('lambda_Car_5', 1.0),
 ('lambda_Distance_0', 1.0),
 ('lambda_Distance_1', 0.4),
 ('lambda_Distance_2', 0.3),
 ('lambda_Distance_3', 0.2),
 ('lambda_Distance_4', 0.1),
 ('lambda_Distance_5', 0.0),
 ('lambda_Landsize_0', 0.0),
 ('lambda_Landsize_1', 0.107556),
 ('lambda_Landsize_2', 0.207556),
 ('lambda_Landsize_

In [ ]:
min_max = []
for i in range(len(data.columns)):
    mini = 
    maxi = ...
    min_max.append((mini, maxi))